# Scale Detection & Recognition — Error Analysis (RIG_images)

This notebook performs error analysis on the scale-detection results for the **RIG_images**
dataset. The detection pipeline writes one JSON per image into `results/`, so the notebook
**first merges every `results/*.json` into a single `results/scale_detection_results.csv`**
and then runs the analysis on that table. It checks:

1. **Completeness** — was there a result for every image in `images/`?
2. **Errors** — did any image return an `error` message?
3. **Inconsistencies** — did any image have a *recognised* scale (length / units / ratio /
   non-zero confidence / `scale_length_flag`) while `scale_bar_found == False`?
   That would mean something was detected/recognised but not flagged as found.
4. **Statistics** — how many scale bars were detected vs. total, and for images with
   scale bars, distributions of the measured quantities and confidences.
5. **Plots** — to spot outliers and patterns.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Notebook lives in RIG_images/. The per-image result JSONs are in results/ and the
# source images (one flat folder) are in images/, both relative to it.
HERE = Path.cwd()
if HERE.name != "RIG_images" and (HERE / "RIG_images").is_dir():
    HERE = HERE / "RIG_images"
RESULTS_DIR = HERE / "results"          # one <image_name>.json per image
IMAGES_DIR = HERE / "images"            # source images (flat)
RESULTS_CSV = RESULTS_DIR / "scale_detection_results.csv"   # built below
print("Results JSON dir:", RESULTS_DIR)
print("Images dir      :", IMAGES_DIR)
print("Merged CSV       :", RESULTS_CSV)
assert RESULTS_DIR.is_dir(), RESULTS_DIR

In [ ]:
# --- Build one merged CSV from the per-image result JSONs -------------------
# The pipeline writes results/<image_name>.json (one per image, see scaledetection.py),
# where <image_name> is the source image's filename without extension. Most images are
# .jpg, but a few are .jpeg, so we resolve the real file on disk per stem rather than
# assuming an extension. We load every JSON, attach image_file / image_path / error,
# and write a single results/scale_detection_results.csv.
IMG_EXTS = [".jpg", ".jpeg", ".png", ".tif", ".tiff"]

# Map each image stem -> its actual filename on disk (handles .jpg vs .jpeg, etc.).
stem_to_name = {}
if IMAGES_DIR.is_dir():
    for p in IMAGES_DIR.iterdir():
        if p.suffix.lower() in IMG_EXTS:
            stem_to_name[p.name[: -len(p.suffix)]] = p.name

json_paths = sorted(RESULTS_DIR.glob("*.json"))
print(f"Found {len(json_paths):,} result JSON files in {RESULTS_DIR}")

records = []
bad = []
for jp in json_paths:
    try:
        with open(jp) as f:
            rec = json.load(f)
        load_err = np.nan
    except Exception as e:  # corrupt/unreadable JSON -> record as an error row
        rec, load_err = {}, f"{type(e).__name__}: {e}"
        bad.append(jp.name)
    image_name = jp.stem                       # e.g. "ABCMB11125-24.6c8b7c6b"
    # Use the real on-disk filename when we can find it; fall back to .jpg.
    fname = stem_to_name.get(image_name, f"{image_name}.jpg")
    rec["image_file"] = fname
    rec["image_path"] = str(IMAGES_DIR / fname)
    # Pipeline JSONs have no "error" field; surface load failures here, else NaN.
    rec.setdefault("error", load_err)
    records.append(rec)

if bad:
    print(f"⚠️  {len(bad)} JSON file(s) could not be parsed; see 'error' column.")

df = pd.DataFrame(records)

# Stable, readable column order: identifiers first, then the pipeline fields.
lead = ["image_file", "image_path", "scale_bar_found", "type_"]
lead = [c for c in lead if c in df.columns]
rest = [c for c in df.columns if c not in lead and c != "error"]
df = df[lead + rest + ["error"]]

RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RESULTS_CSV, index=False)
print(f"Wrote {len(df):,} rows x {df.shape[1]} cols -> {RESULTS_CSV}")

In [ ]:
# Reload from the merged CSV so dtypes/NaNs are normalised the same way every run.
df = pd.read_csv(RESULTS_CSV, low_memory=False)
print(f"{len(df):,} rows  x  {df.shape[1]} columns")
df.head()

In [ ]:
# Column types and null counts at a glance.
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_null": df.isna().sum(),
    "pct_null": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True),
})
summary

## 1. Completeness — was every image processed?

The canonical work list is every image file in `images/`. Each processed image should have
produced exactly one `results/<image_name>.json` (and therefore one row in the merged CSV).
We compare the set of `image_file`s in the results against the images on disk, in both
directions, to catch images that were never processed and any result rows with no matching
image file.

In [ ]:
# Duplicates would corrupt every downstream count, so check first.
n_dupe = df["image_file"].duplicated().sum()
print(f"Duplicate image_file rows: {n_dupe:,}")
if n_dupe:
    display(df[df["image_file"].duplicated(keep=False)].sort_values("image_file").head(20))

In [ ]:
# The canonical expected list is simply every image file in images/ (flat folder).
exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
if IMAGES_DIR.is_dir():
    expected = {p.name for p in IMAGES_DIR.iterdir() if p.suffix.lower() in exts}
    print(f"Image files on disk in {IMAGES_DIR}: {len(expected):,}")
else:
    expected = set()
    print(f"⚠️  Images dir not found ({IMAGES_DIR}); completeness check vs disk skipped.")

In [ ]:
processed = set(df["image_file"])

not_processed = expected - processed   # on disk but absent from results
extra = processed - expected           # in results but no image file on disk

print(f"Expected images       : {len(expected):,}")
print(f"Processed (in results): {len(processed):,}")
print(f"Missing from results  : {len(not_processed):,}")
print(f"Result rows w/o image : {len(extra):,}")

if not_processed:
    print("\nFirst few NOT processed:")
    for name in sorted(not_processed)[:10]:
        print("  ", name)
if extra:
    print("\nFirst few result rows with no matching image file:")
    for name in sorted(extra)[:10]:
        print("  ", name)

if expected and not not_processed and not extra:
    print("\n✅ Every image on disk was processed exactly once.")

In [ ]:
# Cross-check the raw artifacts: number of result JSONs vs images on disk vs CSV rows.
# These three should all agree if every image was processed exactly once.
n_json = len(list(RESULTS_DIR.glob("*.json")))
n_imgs = len(expected)
n_rows = len(df)
print(f"Result JSON files : {n_json:,}")
print(f"Images on disk    : {n_imgs:,}")
print(f"Rows in merged CSV: {n_rows:,}")
if n_json == n_rows and (not expected or n_imgs == n_rows):
    print("✅ JSON count, image count and CSV rows all agree.")
else:
    print("⚠️  Counts disagree — investigate not_processed / extra above.")

## 2. Errors — did any image return an `error` message?

In [ ]:
err_mask = df["error"].notna() & (df["error"].astype(str).str.strip() != "")
n_err = int(err_mask.sum())
print(f"Rows with an error message: {n_err:,}  ({n_err / len(df) * 100:.3f}%)")

if n_err:
    print("\nMost common error messages:")
    display(df.loc[err_mask, "error"].value_counts().head(20))
    print("\nSample rows with errors:")
    display(df.loc[err_mask, ["image_file", "scale_bar_found", "type_", "error"]].head(20))
else:
    print("✅ No image returned an error message.")

## 3. Inconsistencies — recognised content but `scale_bar_found == False`

Flag any row where **something was recognised** yet the scale bar was reported as
*not found*. "Recognised" means any of:

- a non-null `measured_scale_length`, `declared_scale_length`, `units`, or `pixel_to_mm_ratio`; **or**
- a non-zero `scale_bar_confidence` or `text_label_confidence`; **or**
- `scale_length_flag == True`.

Combined with `scale_bar_found == False`, this means something was detected/recognised
but not recorded as a found scale bar.

In [ ]:
def _num(col):
    """Coerce a column to numeric (non-numeric -> NaN)."""
    return pd.to_numeric(df[col], errors="coerce")

not_null_fields = ["measured_scale_length", "declared_scale_length", "units", "pixel_to_mm_ratio"]
has_not_null = df[not_null_fields].notna().any(axis=1)

conf_nonzero = (_num("scale_bar_confidence").fillna(0) != 0) | (
    _num("text_label_confidence").fillna(0) != 0
)

length_flag = df["scale_length_flag"].astype(str).str.strip().str.lower().isin({"true", "1"})

recognised = has_not_null | conf_nonzero | length_flag

found = df["scale_bar_found"].astype(str).str.strip().str.lower().isin({"true", "1"})

inconsistent = recognised & ~found
n_inc = int(inconsistent.sum())
print(f"Rows recognised-but-not-found: {n_inc:,}  ({n_inc / len(df) * 100:.3f}%)")

In [ ]:
if n_inc:
    inc = df[inconsistent].copy()
    # Which signal(s) tripped the flag, per row, for diagnosis.
    inc_reasons = pd.DataFrame({
        "not_null_field": has_not_null[inconsistent],
        "nonzero_confidence": conf_nonzero[inconsistent],
        "scale_length_flag": length_flag[inconsistent],
    })
    print("How many inconsistent rows are tripped by each signal:")
    display(inc_reasons.sum().to_frame("n_rows"))
    cols = ["image_file", "scale_bar_found"] + not_null_fields + [
        "scale_bar_confidence", "text_label_confidence", "scale_length_flag"
    ]
    display(inc[cols].head(30))
else:
    print("✅ No recognised-but-not-found inconsistencies.")

In [ ]:
# Build a folder of images with inconsistencies for manual review, and save a CSV with the relevant metadata for each.
ERROR_FOLDER = HERE / "error_analysis" / "error_images"
ERROR_FOLDER.mkdir(parents=True, exist_ok=True)

if n_err or n_inc:
    err_df = df[err_mask | inconsistent].copy()
    err_df.to_csv(ERROR_FOLDER / "errors.csv", index=False)
    print(f"Saved {len(err_df):,} error rows to {ERROR_FOLDER / 'errors.csv'}")
    for img_path, img_name in zip(err_df["image_path"], err_df["image_file"]):
        src = Path(img_path)
        dst = ERROR_FOLDER / img_name
        if dst.exists() or dst.is_symlink():
            dst.unlink()  # refresh stale link from a previous run
        if src.exists():
            dst.symlink_to(src)
        else:
            print(f"Warning: source image not found for {img_name}: {src}")
else:
    print("No errors to save or link.")

## 4. Statistics

### 4a. Detection rate

In [ ]:
n_total = len(df)
n_found = int(found.sum())
print(f"Total images           : {n_total:,}")
print(f"Scale bar found        : {n_found:,}  ({n_found / n_total * 100:.2f}%)")
print(f"Scale bar NOT found    : {n_total - n_found:,}  ({(n_total - n_found) / n_total * 100:.2f}%)")
print()
print("type_ breakdown:")
display(df["type_"].value_counts(dropna=False).to_frame("n"))
print("Detection rate by type_:")
display(
    df.assign(found=found)
      .groupby("type_")["found"]
      .agg(n="size", n_found="sum", rate=lambda s: f"{s.mean()*100:.2f}%")
)

In [ ]:
# Subset of images where a scale bar WAS found — the rest of the stats focus here.
found_df = df[found].copy()
for c in ["measured_scale_length", "declared_scale_length", "pixel_to_mm_ratio",
          "scale_bar_confidence", "text_label_confidence", "orientation_confidence"]:
    found_df[c] = pd.to_numeric(found_df[c], errors="coerce")
print(f"Images with scale bar found: {len(found_df):,}")
print()
print("Field availability among found scale bars:")
avail = pd.DataFrame({
    "n_present": found_df[["measured_scale_length", "declared_scale_length", "units",
                            "pixel_to_mm_ratio"]].notna().sum(),
    "pct_present": (found_df[["measured_scale_length", "declared_scale_length", "units",
                              "pixel_to_mm_ratio"]].notna().mean() * 100).round(2),
})
display(avail)

### 4b. Numeric distributions (scale bars only)

In [ ]:
num_cols = ["measured_scale_length", "declared_scale_length", "pixel_to_mm_ratio",
            "scale_bar_confidence", "text_label_confidence", "orientation_confidence"]
display(found_df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

In [ ]:
# Units / declared label values among found scale bars.
print("Units distribution:")
display(found_df["units"].value_counts(dropna=False).head(20).to_frame("n"))
print("scale_length_flag among found:")
display(found_df["scale_length_flag"].value_counts(dropna=False).to_frame("n"))

## 5. Plots — spotting outliers and patterns

In [ ]:
# Detected vs not-detected overview.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = pd.Series({"found": n_found, "not found": n_total - n_found})
axes[0].bar(counts.index, counts.values, color=["#2a9d8f", "#e76f51"])
axes[0].set_title("Scale bar detection")
axes[0].set_ylabel("images")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}\n({v/n_total*100:.1f}%)", ha="center", va="bottom")

axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=["#2a9d8f", "#e76f51"], startangle=90)
axes[1].set_title("Detection share")
plt.tight_layout()
plt.show()

In [ ]:
# Confidence distributions (found scale bars).
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(
    axes,
    ["scale_bar_confidence", "text_label_confidence", "orientation_confidence"],
    ["#264653", "#2a9d8f", "#e9c46a"],
):
    data = found_df[col].dropna()
    ax.hist(data, bins=40, color=color, edgecolor="white")
    ax.set_title(f"{col}\n(median={data.median():.3f})")
    ax.set_xlabel(col)
    ax.set_ylabel("images")
plt.tight_layout()
plt.show()

In [ ]:
# Scale-length distributions. These can be heavy-tailed, so show raw + log scale.
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for row, col in enumerate(["measured_scale_length", "declared_scale_length"]):
    data = found_df[col].dropna()
    data = data[np.isfinite(data)]
    axes[row, 0].hist(data, bins=60, color="#457b9d", edgecolor="white")
    axes[row, 0].set_title(f"{col} (linear)  n={len(data):,}")
    axes[row, 0].set_xlabel(col)
    pos = data[data > 0]
    axes[row, 1].hist(pos, bins=np.logspace(np.log10(pos.min()), np.log10(pos.max()), 60)
                      if len(pos) else 60, color="#1d3557", edgecolor="white")
    axes[row, 1].set_xscale("log")
    axes[row, 1].set_title(f"{col} (log, >0 only)  n={len(pos):,}")
    axes[row, 1].set_xlabel(col)
plt.tight_layout()
plt.show()

In [ ]:
# pixel_to_mm_ratio — the calibration output; outliers here matter most downstream.
ratio = found_df["pixel_to_mm_ratio"].dropna()
ratio = ratio[np.isfinite(ratio)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(ratio, bins=60, color="#6a4c93", edgecolor="white")
axes[0].set_title(f"pixel_to_mm_ratio (linear)  n={len(ratio):,}")
axes[0].set_xlabel("pixel_to_mm_ratio")
pos = ratio[ratio > 0]
if len(pos):
    axes[1].hist(pos, bins=np.logspace(np.log10(pos.min()), np.log10(pos.max()), 60),
                 color="#6a4c93", edgecolor="white")
    axes[1].set_xscale("log")
axes[1].set_title(f"pixel_to_mm_ratio (log, >0)  n={len(pos):,}")
axes[1].set_xlabel("pixel_to_mm_ratio")
plt.tight_layout()
plt.show()

# Tukey-fence outlier flag for the ratio.
if len(ratio):
    q1, q3 = ratio.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = found_df[(found_df["pixel_to_mm_ratio"] < lo) | (found_df["pixel_to_mm_ratio"] > hi)]
    print(f"pixel_to_mm_ratio IQR fence: [{lo:.4g}, {hi:.4g}] -> {len(out):,} outliers")
    display(out.sort_values("pixel_to_mm_ratio")[
        ["image_file", "measured_scale_length", "declared_scale_length", "units",
         "pixel_to_mm_ratio", "scale_bar_confidence", "text_label_confidence"]
    ].head(15))

In [ ]:
# Confidence relationship: does scale-bar confidence track text-label confidence?
sub = found_df[["scale_bar_confidence", "text_label_confidence"]].dropna()
if len(sub):
    plt.figure(figsize=(6, 6))
    plt.hexbin(sub["scale_bar_confidence"], sub["text_label_confidence"],
               gridsize=40, cmap="viridis", mincnt=1)
    plt.colorbar(label="images")
    plt.xlabel("scale_bar_confidence")
    plt.ylabel("text_label_confidence")
    plt.title("Scale-bar vs text-label confidence (found scale bars)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Measured vs declared scale length — they should broadly agree; off-diagonal = suspect.
sub = found_df[["measured_scale_length", "declared_scale_length"]].dropna()
sub = sub[(sub > 0).all(axis=1) & np.isfinite(sub).all(axis=1)]
if len(sub):
    plt.figure(figsize=(6, 6))
    plt.scatter(sub["declared_scale_length"], sub["measured_scale_length"],
                s=6, alpha=0.2, color="#e76f51")
    lim = [min(sub.min()), max(sub.max())]
    plt.plot(lim, lim, "--", color="k", lw=1, label="y = x")
    plt.xscale("log"); plt.yscale("log")
    plt.xlabel("declared_scale_length")
    plt.ylabel("measured_scale_length")
    plt.title(f"Measured vs declared scale length  n={len(sub):,}")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Not enough rows with both measured and declared lengths to plot.")

In [ ]:
# Units composition among found scale bars.
uc = found_df["units"].value_counts(dropna=False).head(12)
if len(uc):
    plt.figure(figsize=(8, 4))
    uc[::-1].plot(kind="barh", color="#2a9d8f")
    plt.title("Units among found scale bars (top 12)")
    plt.xlabel("images")
    plt.tight_layout()
    plt.show()

## Summary

Re-run the cell below after executing the notebook to get a one-glance verdict.

In [ ]:
print("=" * 60)
print("SCALE DETECTION ERROR-ANALYSIS SUMMARY")
print("=" * 60)
print(f"Rows in results            : {len(df):,}")
print(f"Duplicate image_file rows  : {n_dupe:,}")
print(f"Expected images            : {len(expected):,}")
print(f"Missing from results       : {len(not_processed):,}")
print(f"Unexpected extra rows      : {len(extra):,}")
print(f"Rows with error message    : {n_err:,}")
print(f"Recognised-but-not-found   : {n_inc:,}")
print(f"Scale bar found            : {n_found:,}  ({n_found/len(df)*100:.2f}%)")
print("=" * 60)